In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from scipy.stats import norm

data = pd.read_csv("train.csv")
data['date'] = pd.to_datetime(data['date'])

In [3]:
store_id = 1
item_id = 1

df = data[(data['store']==store_id) & (data['item']==item_id)].copy()
df = df.sort_values('date')

In [4]:
df['lag_1'] = df['sales'].shift(1)
df['lag_7'] = df['sales'].shift(7)
df['lag_30'] = df['sales'].shift(30)

df['rolling_7'] = df['sales'].shift(1).rolling(7).mean()
df['rolling_30'] = df['sales'].shift(1).rolling(30).mean()

df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.dayofweek

df = df.dropna()

In [5]:
features = ['lag_1','lag_7','lag_30','rolling_7','rolling_30','month','day_of_week']

split = int(len(df)*0.8)

train = df.iloc[:split]
valid = df.iloc[split:]

X_train = train[features]
y_train = train['sales']

X_valid = valid[features]
y_valid = valid['sales']

model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

model.fit(X_train,y_train)

predictions = model.predict(X_valid)

In [6]:
forecast_mean = np.mean(predictions)

residuals = y_valid - predictions
forecast_std = np.std(residuals)

print("Predicted Daily Demand:",forecast_mean)
print("Demand Uncertainty:",forecast_std)

Predicted Daily Demand: 21.362328
Demand Uncertainty: 5.174925227010631


In [7]:
lead_time = 7
service_level = 0.95

z = norm.ppf(service_level)

safety_stock = z * forecast_std * np.sqrt(lead_time)
reorder_point = forecast_mean * lead_time + safety_stock

print("Safety Stock:",safety_stock)
print("Reorder Point:",reorder_point)

Safety Stock: 22.520620684482438
Reorder Point: 172.05690608487305
